# Etapa 5n — Cierre de la versión 2 del algoritmo

**Qué es.** La versión 2 junta las mejoras adoptadas en la ronda para la entrega final (desvío 14.11 del plan), todas decididas solo con desarrollo y con la regla escrita antes de correr:

| Etapa | Cambio | Matthews en desarrollo (validación cruzada) |
|---|---|---|
| — | Versión 1, congelada (`congelado-test`): resta del fondo, 26 mapas | 0,7302 |
| 5g | Normalización del fondo por división | 0,7509 |
| 5h | Aperturas por reconstrucción (28 mapas) | 0,7520 |
| 5i | Relleno de exudados | 0,7533 |
| 5k | Wiener con el ruido estimado (por robustez: con ruido 10, de 0,648 a 0,712) | 0,7525 |
| 5l | B-COSFIRE (31 mapas) | 0,7527 |
| 5m | Selección de mapas con L1 (13 a 15 mapas) | 0,7502 |

No se adoptaron: la histéresis (5j), el borde asimétrico (5h), la simetría de fase, las aperturas por caminos y el hessiano (5l).

**Cómo se ajusta** (como la versión 1 en la etapa 8, con todas las imágenes de desarrollo):
1. Píxeles de entrenamiento de las 30 imágenes (entrenamiento equilibrado por grosor) con los 31 mapas.
2. Regresión con L1 (`pipeline.fit_weights_l1`): λ por validación cruzada con los 5 grupos de personas y la regla de un error estándar, y ajuste con las 30 imágenes. Se guardan solo los mapas con peso distinto de cero; los demás no se calculan.
3. Valor fijo del umbral con las 30 imágenes (`choose_value`), y el de la referencia clásica (detector de líneas solo) con la misma regla.
4. Se guarda en `models/pesos_v2.json`, con su preprocesamiento y su lista de mapas. **La versión 1 (`models/pesos_combinacion.json`) no cambia y sigue siendo la oficial:** es la que se evaluó en prueba.

**Qué estimación vale.** La de validación cruzada de `05m` (Matthews 0,7502, vasos finos 33 %, Dice de línea central 0,752), con la salvedad de que es optimista: siete rondas se decidieron con las mismas 30 imágenes. Medir esta versión con las imágenes con que se ajustó no dice nada nuevo, así que acá solo se comprueba que funciona. El conjunto de prueba no se usa.

In [ ]:
import sys
import time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src import config as C
from src.data import dev_ids, load_image, load_split
from src.evaluation import map_images
from src.pipeline import (WEIGHTS_PATH, WEIGHTS_V2_PATH, choose_value, fit_weights_l1, load_feature_names,
                          save_reference_value, save_weights, segment_final)
from src.split import person_folds

pd.set_option("display.width", 220)
ids = dev_ids()
dev = load_split().set_index("image").loc[ids]
folds = dev["person"].map(person_folds(dev.reset_index(), 5, C.SEED))
MODE = C.SEGMENT_BACKGROUND_MODE
print(MODE, "|", len(C.SEGMENT_FEATURES), "mapas candidatos | ajuste:", C.SEGMENT_FIT)

## Píxeles de entrenamiento de las 30 imágenes

In [ ]:
def pass1(image_id):
    from src.pipeline import training_pixels
    return image_id, training_pixels(image_id, background=MODE)

t = time.time()
samples = dict(map_images(pass1, ids, workers=4))
print(f"Pasada 1: {time.time() - t:.0f} s")

## Ajuste con L1 y valores fijos

In [ ]:
t = time.time()
w, lam, info = fit_weights_l1(samples, ids, {i: int(folds[i]) for i in ids})
print(f"Ajuste: {time.time() - t:.0f} s | λ = {lam:.5f} (el de menor devianza: {info['best']:.5f})")
keep = np.flatnonzero(w[1:])
names = [C.SEGMENT_FEATURES[j] for j in keep]
w_v2 = np.r_[w[0], w[1:][keep]]
sub = {i: {k: (v[:, keep] if k in ("x", "x_thr") else v) for k, v in s.items()} for i, s in samples.items()}
value = choose_value(sub, w_v2, ids)
ref_value = choose_value(samples, None, ids)
print(f"{len(names)} mapas | valor fijo {value:.4f} | referencia {ref_value:.4f}")
pd.Series(w_v2[1:], index=names, name="peso").sort_values(key=np.abs, ascending=False).round(3)

## Se guarda en `models/pesos_v2.json`

In [ ]:
save_weights(w_v2, WEIGHTS_V2_PATH, value=value, background=MODE, feature_names=names,
             note=("Versión 2 (etapas 5g a 5m, desvío 14.11): Wiener con ruido estimado, relleno de exudados y "
                   "división; regresión con L1 sobre 31 mapas, λ por validación cruzada por personas con la regla "
                   f"de un error estándar (λ = {lam:.5f}); ajustada con las 30 imágenes de desarrollo "
                   "(notebooks/05n_version2.ipynb). No se evaluó en prueba."))
save_reference_value(ref_value, WEIGHTS_V2_PATH)
assert load_feature_names(WEIGHTS_V2_PATH) == names
print(WEIGHTS_V2_PATH.read_text(encoding="utf-8")[:600])

## Comprobación: funciona con `segment_final` y cuánto tarda (3 imágenes de desarrollo)

In [ ]:
rows = []
for image_id in ids[:3]:
    rgb = load_image(image_id)
    for version, path in (("v1", WEIGHTS_PATH), ("v2", WEIGHTS_V2_PATH)):
        t = time.time()
        m = segment_final(rgb, path=path)
        rows.append({"imagen": image_id, "versión": version, "segundos": round(time.time() - t, 1),
                     "fracción marcada": round(float(m.mean()), 4)})
    t = time.time()
    r = segment_final(rgb, reference=True, path=WEIGHTS_V2_PATH)
    rows.append({"imagen": image_id, "versión": "v2 referencia", "segundos": round(time.time() - t, 1),
                 "fracción marcada": round(float(r.mean()), 4)})
pd.DataFrame(rows)

## Resumen (escrito después de correr)

*(pendiente)*